# Prueba de funcionlidad `procesarOT.py`

### Cargar Librerias

In [22]:
from datetime import datetime

In [ ]:
import os
import sys
from time
from datetime import datetime
import pymongo
from pymongo.errors import ConnectionFailure
import logging

logging.basicConfig(level=logging.INFO)

from eerssa.secret import Keys

# --- MongoDB Connection ---
# It's better to establish the connection once and keep it open for the app's lifetime.
# We will also exit if the connection fails, as the consumer can't do its job without it.
uri = Keys.MONGO_KEY.value
client = None  # Initialize client to None
db_eerssa = None
CurrentCollection = None
ReloadCollection = None


try:
    # Add a timeout to avoid blocking indefinitely
    client = pymongo.MongoClient(uri, serverSelectionTimeoutMS=5000)
    # The ping command is cheap and does not require auth.
    client.admin.command('ping')
    db_eerssa = client.eerssa                   # Base de datos EERSSA
    CurrentCollection = db_eerssa.ot_v30        # Coleccion actual
    logging.info(":::: Conexion exitosa con MongoDB ::::")
    
except ConnectionFailure as e:
    logging.error(f"\n\n ><><> Error de conexion a MongoDB: {e}")
    sys.exit(1) # Exit the script if we can't connect to MongoDB, as it's a critical dependency.


INFO:root::::: Conexion exitosa con MongoDB ::::


In [ ]:
# DASK

from dask.distributed import LocalCluster, as_completed
dask = LocalCluster().get_client()
visor_dask = dask.dashboard_link

INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/.venv/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44649 instead
  warnings.warn(
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:32921
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:44649/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42363'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42811'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:35987'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:37151'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:4120

'http://127.0.0.1:44649/status'

In [3]:
## RECARGAR LAS LIBRERIAS DINAMICAMENTE
from importlib import reload
from eerssa import procesarOt as OrdenTrabajo             # Convert from PDF_ot to obj_ot
from eerssa import generarMatrizActividades as Actividades     # process ot.data["actividades"]

### Recargar Librerias

In [5]:
reload( OrdenTrabajo )
reload( Actividades  )
print(visor_dask)

http://127.0.0.1:44649/status


### Directorios de Prueba

In [16]:
from pathlib import Path

dir_test = Path ("/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test")

# Get all entries (files and subdirectories)
all_entries = dir_test.iterdir()

# Filter for only files
files = [item for item in all_entries if item.is_file()]

# You can also get just the names if you prefer
# file_names = [item.name for item in all_entries if item.is_file()]

print("All files in the directory (Path objects):")
print(files)

# If you need them as strings
file_strings = [str(f) for f in files]
print("\nAll files as strings:")
print(file_strings)

All files in the directory (Path objects):
[PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/28_6 colaboradores.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/7_one_line_text_overlap.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_LM_Tres_hojas.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_casoEspecial01.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/14_Orden de trabajo Guayzimi 10-04-2022 (CQ).pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/27_5 colaboradores.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/8_bad_line_text_overlap.pdf'), PosixPath('/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/21_0 colaboradores y vehiculo.pdf'), PosixPath('/

In [18]:
# Verificar la conversion de archivos
list_pdfs = file_strings
print(list_pdfs)

['/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/10_Test Orden de trabajo Zamora 11-02-2022 (RM - Electricistas).pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_LM_Tres_hojas.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_accidente canoa SI.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/12_casoEspecial01.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/13 En una actividad falta la hora Final.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/14_Orden de trabajo Guayzimi 10-04-2022 (CQ).pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/1_not_pdf.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/20_0 colaboradores sin vehiculo.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/21_0 colaboradores y vehiculo.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/22_1 colaborador sin vehiculo.pdf', '/home/vlad/GIT/eerssa_gh/ordenes_de_trab

### DASK Parallel Computing

In [23]:
# Verificar la conversion de archivos
list_pdfs = file_strings

print(f" Se han encontrado un total de: {len(list_pdfs)} Ordenes de Trabajo" )

start_time = time.time()
start_datetime = datetime.now()
print( f"Hora de inicio: {start_datetime.strftime('%Y-%m-%d %H:%M:%S')}\n\n" )

# 1. Submit the first batch of tasks
# This returns a list of futures, same as before.
futures_step1 = [dask.submit(OrdenTrabajo.procesarOt, file) for file in list_pdfs]

# 2. Submit the second batch of tasks, feeding the first futures as input
#futures_step2 = [dask.submit(call_load_ot, f) for f in futures_step1]

# 3. Now, gather only the FINAL results
# This single call executes the entire graph (both GestionOt and load_ot) in parallel.
obj_lists_dask = dask.gather(futures_step1)


end_time = time.time()
elapsed_time = end_time - start_time
end_datetime = datetime.now()
print(f"\n\n   Procesados todos los {len(obj_lists_dask)} items. Tiempo transcurrido: {elapsed_time:.2f} segundos.\n   Hora Final : {end_datetime.strftime('%Y-%m-%d %H:%M:%S')}")



 Se han encontrado un total de: 28 Ordenes de Trabajo
Hora de inicio: 2025-08-02 17:43:15


Success!!!
Success!!!
Success!!!
Success!!!


   Procesados todos los 28 items. Tiempo transcurrido: 0.35 segundos.
   Hora Final : 2025-08-02 17:43:16


In [25]:
obj_lists_dask[6]

{'version': '0.3.0',
 'link': '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/1_not_pdf.pdf',
 'exito': False,
 'log': [{'t': datetime.datetime(2025, 8, 2, 17, 43, 15, 956135),
   'level': 'FATAL',
   'message': 'No es un archivo PDF',
   'detail': 'No se reconoce como archivo PDF valido: /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/1_not_pdf.pdf'}]}